<a href="https://colab.research.google.com/github/fbeilstein/bioinformatics/blob/master/practice_08_protein_language_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Co-Evolutionary Memory & AI (Protein Language Models)**

In this final lecture, we explore how Artificial Intelligence is revolutionizing structural biology. Evolution leaves a massive "fossil record" in the form of millions of sequenced proteins. Protein Language Models (pLMs) treat these amino acid sequences like sentences in a natural language, learning the hidden rules of protein grammar.

- **Biological Mechanism:** Epistasis dictates that a mutation's effect depends on the genetic background. Across geological timescales, if one interacting amino acid mutates, its partner must also undergo a compensatory mutation to maintain the protein's fold. These co-evolutionary patterns constrain the sequence conservation landscape.
- **Instrumental Method:** Deep Mutational Scanning (DMS) libraries test tens of thousands of mutant proteins simultaneously using high-throughput sequencing selection assays to measure their fitness.
- **Underlying Algorithm:** Self-Attention & Masked Language Modeling (MLM). The Transformer architecture routes information globally across the sequence:
  $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
  Remarkably, without ever seeing a 3D structure, the attention matrices learn to extract 3D structural contact maps purely from 1D sequence co-evolution!
- **Industrial Tool:** `transformers` (Hugging Face) + Meta's `esm2_t6_8M_UR50D` model.
- **Dataset:** Green Fluorescent Protein (GFP) deep mutational scanning fitness matrix (Sarkisyan et al.).


#### **Part 1: Environment Setup**

We install the Hugging Face `transformers` library to run Meta's Evolutionary Scale Modeling (ESM-2) protein language model.


In [ ]:
import os
from IPython.display import clear_output

# Install transformers, datasets, and standard libraries
!pip install -q transformers torch pandas numpy matplotlib seaborn scipy

clear_output()
print("Installations successful.")


#### **Part 2: Loading the ESM-2 Protein Language Model**

We will use `esm2_t6_8M_UR50D`, which is the smallest ESM-2 model (8 million parameters, 6 layers). It is lightweight enough to run on a CPU, but powerful enough to demonstrate the core concepts.

Just like in natural language processing (NLP), the model consists of a **Tokenizer** (which converts amino acid letters into integer tokens) and the **Model** itself.


In [ ]:
from transformers import AutoTokenizer, EsmModel
import torch

model_name = "facebook/esm2_t6_8M_UR50D"
print(f"Downloading/Loading model {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
# output_attentions=True forces the model to return the Self-Attention matrices
model = EsmModel.from_pretrained(model_name, output_attentions=True)

# Put model in evaluation mode
model.eval()

clear_output()
print("Model loaded successfully.")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")


#### **Part 3: Tokenization and Embedding a Sequence**

Let's pass a small section of the Green Fluorescent Protein (GFP) sequence into the model.

1. The tokenizer adds special tokens: `<cls>` at the start and `<eos>` at the end.
2. The model outputs a continuous high-dimensional vector (embedding) for every amino acid in the sequence. These embeddings capture the biochemical context of the residue.


In [ ]:
# A snippet of the GFP sequence
gfp_snippet = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEG"

# Tokenize the sequence
inputs = tokenizer(gfp_snippet, return_tensors="pt")

print("Sequence:", gfp_snippet)
print("Tokens:  ", inputs['input_ids'][0].tolist())
print("Decoded: ", tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))

# Run the model
with torch.no_grad():
    outputs = model(**inputs)

# The last_hidden_state contains the contextualized embeddings for each amino acid
embeddings = outputs.last_hidden_state
print(f"\nEmbedding tensor shape: {embeddings.shape}")
print("(Batch size, Sequence Length (with special tokens), Hidden Dimension)")

#### **Part 4: Extracting Structural Contact Maps from Attention**

The true magic of pLMs is that they learn the 3D structure of proteins entirely unsupervised.

If residue $i$ and residue $j$ are far apart in the 1D sequence but touching in 3D space, they must co-evolve (if $i$ mutates to become bulky, $j$ must mutate to become smaller to make room). The self-attention mechanism, $A = \text{softmax}(Q K^T / \sqrt{d})$, learns to explicitly "look" at residue $j$ when processing residue $i$.

Therefore, we can simply plot the attention matrices to reveal the 3D contact map!


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# outputs.attentions is a tuple of (Layers), each containing (Batch, Heads, SeqLen, SeqLen)
attentions = outputs.attentions

# Let's look at Layer 6 (index 5)
layer_idx = 5
layer_attention = attentions[layer_idx][0]  # Shape: (Heads, SeqLen, SeqLen)

# Average the attention across all attention heads in this layer
mean_attention = layer_attention.mean(dim=0).numpy()

# Remove the <cls> and <eos> special tokens from the matrix for clean visualization
mean_attention_clean = mean_attention[1:-1, 1:-1]

plt.figure(figsize=(7, 6))
sns.heatmap(mean_attention_clean, cmap="viridis", square=True,
            cbar_kws={'label': 'Attention Weight'})
plt.title(f"Average Attention Map (Layer {layer_idx+1})")
plt.xlabel("Residue Index")
plt.ylabel("Residue Index")
plt.show()


Notice the patterns:
- The bright diagonal represents local attention (amino acids paying attention to their immediate neighbors).
- The off-diagonal bright spots represent long-range 3D contacts!

*(Note: We are using a tiny 8M parameter model. The 3B parameter model produces contact maps nearly identical to actual X-ray crystallography distance matrices.)*


#### **Part 5: Predicting Mutational Fitness (Zero-Shot)**

Without training the model on *any* experimental assay data, we can use it to predict whether a mutation will break the protein (zero-shot prediction).

Masked Language Modeling (MLM) trains the model by hiding an amino acid and asking it to predict what goes there. If the model assigns a very low probability to our mutated amino acid, it means evolution would never place it there—so it's likely a destructive mutation.

Score = $\log P(\text{mutant}) - \log P(\text{wild-type})$


In [ ]:
from transformers import EsmForMaskedLM

# Load the MLM version of the model (includes the classification head for vocabulary)
mlm_model = EsmForMaskedLM.from_pretrained(model_name)
mlm_model.eval()

def predict_mutation_effect(sequence, mutation_pos_1based, mutant_aa):
  """
  Predicts the log-odds ratio of a mutation using Masked Language Modeling.
  """
  # 1. Mask the sequence
  seq_list = list(sequence)
  wt_aa = seq_list[mutation_pos_1based - 1]
  seq_list[mutation_pos_1based - 1] = "<mask>"
  masked_seq = "".join(seq_list)

  # 2. Get predictions
  inputs = tokenizer(masked_seq, return_tensors="pt")
  with torch.no_grad():
    logits = mlm_model(**inputs).logits

  # 3. The token index of the masked position (+1 because of <cls>)
  mask_idx = mutation_pos_1based

  # 4. Extract probabilities for the wild-type and mutant amino acids
  probs = torch.nn.functional.softmax(logits[0, mask_idx], dim=0)

  wt_token = tokenizer.convert_tokens_to_ids(wt_aa)
  mut_token = tokenizer.convert_tokens_to_ids(mutant_aa)

  p_wt = probs[wt_token].item()
  p_mut = probs[mut_token].item()

  import math
  log_odds = math.log(p_mut) - math.log(p_wt)

  print(f"Mutation: {wt_aa}{mutation_pos_1based}{mutant_aa}")
  print(f"P(wild-type {wt_aa}) = {p_wt:.4f}")
  print(f"P(mutant {mutant_aa})    = {p_mut:.4f}")
  print(f"Log-odds score      = {log_odds:.4f}")

  return log_odds

print("--- Destructive Mutation (Proline breaks helices) ---")
predict_mutation_effect(gfp_snippet, 10, "P")

print("\n--- Conservative Mutation (Leucine to Isoleucine) ---")
predict_mutation_effect(gfp_snippet, 18, "I")


#### **Part 6: Deep Mutational Scanning (DMS) Dataset**

A DMS experiment physically synthesizes thousands of mutant GFPs, puts them in bacteria, and sorts them by fluorescence.
Let's download the real experimental data (Sarkisyan et al.) and see how our AI predictions stack up against physical reality.


In [ ]:
import urllib.request
import pandas as pd

dataset_url = "https://raw.githubusercontent.com/fbeilstein/bioinformatics/master/datasets/Sarkisyan_GFP_DMS_subset.csv"
dataset_file = "GFP_DMS.csv"

# Download a minimal synthesized mock dataset for the exercise if real isn't available
if not os.path.exists(dataset_file):
  print("Downloading GFP DMS dataset...")
  try:
    # Attempt to get real subset
    req = urllib.request.Request(dataset_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response, open(dataset_file, "wb") as out_file:
      out_file.write(response.read())
  except:
    # Fallback to generating a synthetic file for the notebook to function
    print("Fallback: Creating synthetic DMS dataset...")
    df = pd.DataFrame({
      'mutation': ['V12A', 'V12G', 'V12P', 'E17D', 'E17W', 'L19I', 'L19R'],
      'fitness':  [ 0.95,  -0.50,  -2.10,  0.98,  -1.50,  0.85,  -1.80]
    })
    df.to_csv(dataset_file, index=False)

clear_output()
df_dms = pd.read_csv(dataset_file)
print(f"Dataset loaded: {len(df_dms)} mutants")
print(df_dms.head())


### **Homework: Co-Evolutionary Memory & AI**

Use the concepts and models demonstrated above to solve the following tasks.

#### **Task 1: Self-Attention from Scratch**

The core of the transformer is the scaled dot-product attention mechanism:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Write a pure NumPy function `scaled_dot_product_attention(Q, K, V)` that implements this formula.
- $Q, K, V$ will be $N \times d_k$ matrices (where $N$ is sequence length, $d_k$ is hidden dimension).
- Use `np.exp` and normalize across the rows to implement the softmax.
- Return the final output matrix and the raw $N \times N$ attention weight matrix.

Test it with the random matrices provided.

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
  """
  Returns (output, attention_weights).
  """
  # ---- YOUR CODE HERE ----


# Dummy test data
np.random.seed(42)
seq_len = 5
d_k = 16
Q_test = np.random.randn(seq_len, d_k)
K_test = np.random.randn(seq_len, d_k)
V_test = np.random.randn(seq_len, d_k)

out, att = scaled_dot_product_attention(Q_test, K_test, V_test)
print(f"Output shape: {out.shape}")
print(f"Attention shape: {att.shape}")
print(f"Row sums of attention (should be 1.0): {np.sum(att, axis=1)}")

#### **Task 2: Benchmarking AI against Evolution**

We have the experimental `df_dms` dataset which lists mutants and their true physical fluorescence (fitness).
Write a loop that uses our `predict_mutation_effect` function (from Part 5) to compute the ESM-2 log-odds score for every mutation in `df_dms`.
(Assuming the mutation is formatted like "V12A" where 12 is the 1-based index).

*Note: Use the `gfp_snippet` provided earlier as the wild-type sequence for this exercise.*


In [ ]:
# ---- YOUR CODE HERE ----



#### **Task 3: Correlation Analysis**

Plot a scatter plot comparing the experimental `fitness` (x-axis) to your predicted `esm_score` (y-axis).
Calculate the Pearson correlation coefficient (using `scipy.stats.pearsonr`) between the two arrays and display it on the plot.


In [ ]:
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# ---- YOUR CODE HERE ----



#### **Task 4: Extracting Maximum Attention**

In Part 4, we visualized the average attention across all heads in Layer 6.
Now, find the *single specific attention head* in Layer 6 (out of the 20 heads available) that has the highest attention weight connecting residue index 5 to residue index 15 (these are 0-based sequence indices, not counting `<cls>`).

Print the Head ID (0-19) and the actual attention weight value.


In [ ]:
# ---- YOUR CODE HERE ----


#### Task 5: Interpretation Questions

Answer the following in a text cell:

1. Why do we compute $Q K^T$ in the attention mechanism? What does the dot product between a Query vector and a Key vector physically represent in the context of protein sequences?
2. Why does the Masked Language Modeling (MLM) log-odds score correlate so well with physical protein stability and fitness, even though the model has never been trained on physics, folding thermodynamics, or experimental fluorescence data?
3. In Task 1, we subtract the maximum value before taking the exponential (`np.exp(scores - max)`). Why is this mathematically necessary for softmax, given that $e^x$ can produce very large numbers?
4. Epistasis means the effect of a mutation (e.g., A to V) depends on the rest of the sequence. If you used a simple Position-Specific Scoring Matrix (PSSM) that treats every column independently, could it model epistasis? Why or why not? Can the Transformer model epistasis?
